# 03 Churn Modeling

This notebook trains Logistic Regression, Random Forest, and XGBoost, then compares them using precision, recall, F1-score, and ROC-AUC.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.config import CUSTOMER_FEATURES_PATH, FEATURE_IMPORTANCE_PATH, MODEL_METRICS_PATH, MODEL_PATH
from src.evaluate import classification_report_frame, save_evaluation_figures
from src.feature_importance import extract_feature_importance
from src.train import save_model, train_and_select_model

## Train and Compare Models

The target is `Churn`. `CustomerID` is removed from model inputs because it is an identifier, not behavior.

In [ ]:
features = pd.read_csv(CUSTOMER_FEATURES_PATH)
features['Churn'].value_counts(normalize=True).rename('share')

In [ ]:
selected_model, metrics, context = train_and_select_model(features)
metrics.to_csv(MODEL_METRICS_PATH, index=False)
save_model(selected_model, MODEL_PATH)
metrics

## Model Diagnostics

Accuracy is not the primary metric because churn classes can be imbalanced. Recall matters when the business wants to identify as many at-risk customers as possible.

In [ ]:
classification_report_frame(selected_model, context['X_test'], context['y_test'])

In [ ]:
importance = extract_feature_importance(selected_model, context['feature_names'])
importance.to_csv(FEATURE_IMPORTANCE_PATH, index=False)
importance.head(10)